In [ ]:
import pandas as pd
import numpy as np

# Load the cleaned dataset
cleaned_df = pd.read_csv(
    "../data/processed/cleaned_farmgate_prices.csv",
    parse_dates=["Date"]
)

# Create a separate dataset for modeling
model_data = cleaned_df.copy()

# Verify the dataset
print("Total records:", len(model_data))
print("Total commodities:", model_data["Commodity"].nunique())
print("Total categories:", model_data["Category"].nunique())
print("Missing prices:", model_data["Price"].isna().sum())
print("Interpolated prices:", model_data["Was_Interpolated"].sum())


In [31]:
# 2. Reusable Commodity Preparation Function

# Prepare monthly data for a selected commodity
def prepare_commodity_data(data, category, commodity):
    
    # Filter the selected category and commodity
    commodity_data = data[
        (data["Category"] == category) &
        (data["Commodity"] == commodity)
    ].copy()

    # Sort records by date
    commodity_data = commodity_data.sort_values("Date")

    # Set monthly dates as the index
    commodity_data = commodity_data.set_index("Date").asfreq("MS")
    commodity_data.index = commodity_data.index.to_period("M")

    return commodity_data


# Test using our first commodity
palay_data = prepare_commodity_data(
    model_data,
    "Cereals",
    "Palay [Paddy] Other Variety, dry (conv. to 14% mc)"
)

print("Total months:", len(palay_data))
print("Missing prices:", palay_data["Price"].isna().sum())
print("Interpolated prices:", palay_data["Was_Interpolated"].sum())

display(palay_data.head())

Total months: 202
Missing prices: 0
Interpolated prices: 0


,Geolocation,Category,Commodity,Price,Was_Interpolated
Date,,,,,
2010-01,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",14.82,False
2010-02,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",15.34,False
2010-03,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",15.58,False
2010-04,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",14.92,False
2010-05,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",14.99,False


In [17]:
# 3. Reusable Train-Test Split and Backtesting Windows
from sktime.split import temporal_train_test_split

# Prepare training and testing periods for any commodity
def prepare_train_test(commodity_data):
    monthly_prices = commodity_data["Price"]

    # Chronological 80/20 split
    train_data, test_data = temporal_train_test_split(
        monthly_prices, test_size=0.20
    )

    # Create complete six-month testing windows
    test_windows = []

    for start in range(0, len(test_data) - 5, 6):
        test_window = test_data.iloc[start:start + 6]
        test_windows.append(test_window)

    return monthly_prices, train_data, test_data, test_windows


# Test using our Palay commodity
monthly_prices, train_data, test_data, test_windows = prepare_train_test(palay_data)

print("Training months:", len(train_data))
print("Testing months:", len(test_data))
print("Six-month testing windows:", len(test_windows))


Training months: 161
Testing months: 41
Six-month testing windows: 6


In [ ]:
test_windows

In [ ]:
# 4. Prepare Three Forecasting Models 
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.exp_smoothing import ExponentialSmoothing
from sktime.forecasting.arima import AutoARIMA

# Define the three forecasting models
forecasting_models = {
    "Seasonal Naive": NaiveForecaster(
        strategy="last", sp=12
    ),
    "Holt-Winters": ExponentialSmoothing(
        trend="add", seasonal="add", sp=12
    ),
    "AutoARIMA": AutoARIMA(
        seasonal=False,
        stepwise=True,
        suppress_warnings=True
    )
}

print("Forecasting models:", list(forecasting_models.keys()))


In [53]:
# 5. Reusable Backtesting Function
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Evaluate a forecasting model across six-month testing windows
def backtest_model(model, commodity_data, test_windows):
    backtest_results = []

    for test_window in test_windows:
        # Get historical prices before the testing period
        training_data = commodity_data[
            commodity_data.index < test_window.index[0]
        ]

        training_prices = training_data["Price"]

        # Train a fresh model
        current_model = model.clone()
        current_model.fit(training_prices)

        # Forecast the next 6 months
        predicted_prices = current_model.predict(fh=range(1, 7))

        # Calculate forecasting errors
        backtest_results.append({
            "Test Period": f"{test_window.index[0]} to {test_window.index[-1]}",
            "MAE": mean_absolute_error(test_window, predicted_prices),
            "RMSE": np.sqrt(mean_squared_error(test_window, predicted_prices))
        })

    return pd.DataFrame(backtest_results)

In [86]:
# Backtest Seasonal Naïve
seasonal_results = backtest_model(
    forecasting_models["Seasonal Naive"],
    palay_data,
    test_windows
)

display(seasonal_results.round(2))

,Test Period,MAE,RMSE
0,2023-06 to 2023-11,3.07,3.30
1,2023-12 to 2024-05,6.22,6.26
2,2024-06 to 2024-11,2.79,3.45
3,2024-12 to 2025-05,5.17,5.44
4,2025-06 to 2025-11,6.29,6.53
5,2025-12 to 2026-05,3.34,3.74


In [ ]:
# Backtest Holt-Winters
holt_results = backtest_model(
    forecasting_models["Holt-Winters"],
    palay_data,
    test_windows
)

# Display evaluation results
print("Holt-Winters Results:")
display(holt_results.round(2))

In [51]:
# Backtest AutoARIMA
arima_results = backtest_model(
    forecasting_models["AutoARIMA"],
    palay_data,
    test_windows
)

print("AutoARIMA Results:")
display(arima_results.round(2))

AutoARIMA Results:


,Test Period,MAE,RMSE
0,2023-06 to 2023-11,1.40,1.81
1,2023-12 to 2024-05,2.37,2.49
2,2024-06 to 2024-11,2.24,2.91
3,2024-12 to 2025-05,1.23,1.58
4,2025-06 to 2025-11,1.20,1.33
5,2025-12 to 2026-05,4.46,4.88


In [50]:
# 7. Compare 3 models
# Organize the backtesting results
model_results = {
    "Seasonal Naive": seasonal_results,
    "Holt-Winters": holt_results,
    "AutoARIMA": arima_results
}

# Calculate overall evaluation metrics
evaluation_results = []

for model_name, results in model_results.items():
    evaluation_results.append({
        "Model": model_name,
        "MAE": results["MAE"].mean(),
        "RMSE": np.sqrt((results["RMSE"] ** 2).mean())
    })

# Rank models by lowest MAE
evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df = evaluation_df.sort_values("MAE").reset_index(drop=True)

display(evaluation_df.round(2))

# Identify the best-performing model
best_model = evaluation_df.iloc[0]["Model"]
print("Best-performing model:", best_model)


,Model,MAE,RMSE
0,Holt-Winters,2.12,2.65
1,AutoARIMA,2.15,2.77
2,Seasonal Naive,4.48,4.97


Best-performing model: Holt-Winters


In [61]:

# Evaluate all three forecasting models for one commodity
def evaluate_commodity(data, category, commodity):

    # Prepare the selected commodity
    commodity_data = prepare_commodity_data(data, category, commodity)

    # Create chronological training and testing periods
    monthly_prices, train_data, test_data, test_windows = prepare_train_test(commodity_data)

    evaluation_results = []

    # Evaluate each forecasting model
    for model_name, model in forecasting_models.items():
        results = backtest_model(model, commodity_data, test_windows)

        evaluation_results.append({
            "Model": model_name,
            "MAE": results["MAE"].mean(),
            "RMSE": np.sqrt((results["RMSE"] ** 2).mean())
        })

    # Rank models by lowest MAE
    evaluation_df = pd.DataFrame(evaluation_results)
    return evaluation_df.sort_values("MAE").reset_index(drop=True)


# Test with another commodity: Yellow Corn
corn_results = evaluate_commodity(
    model_data,
    "Cereals",
    "Corngrain [Maize] Yellow, matured"
)

display(corn_results.round(2))


,Model,MAE,RMSE
0,Holt-Winters,1.17,1.46
1,AutoARIMA,1.21,1.63
2,Seasonal Naive,2.11,2.45


In [ ]:

# Store evaluation results and any errors
all_results = []
failed_commodities = []

# Get all unique category and commodity combinations
commodities = model_data[["Category", "Commodity"]].drop_duplicates()

# Evaluate each commodity
for category, commodity in commodities.itertuples(index=False, name=None):
    try:
        results = evaluate_commodity(model_data, category, commodity)

        # Identify which commodity the results belong to
        results["Category"] = category
        results["Commodity"] = commodity

        all_results.append(results)
        print(f"Completed: {commodity}")

    except Exception as error:
        failed_commodities.append((category, commodity, str(error)))
        print(f"Failed: {commodity} - {error}")

# Combine results into one table
all_results_df = pd.concat(all_results, ignore_index=True)

print("\nCommodities evaluated:", len(all_results))
print("Failed commodities:", len(failed_commodities))

display(all_results_df.head(20).round(2))


In [39]:

# Save all model evaluation results
all_results_df.to_csv(
    "../data/processed/model_evaluation_results.csv",
    index=False
)

print("Evaluation results saved successfully!")
print("Total evaluation rows:", len(all_results_df))


Evaluation results saved successfully!
Total evaluation rows: 174


In [59]:

# Rank models by lowest MAE, then RMSE
ranked_results = all_results_df.sort_values(["MAE", "RMSE"])

# Select the best-performing model for each commodity
best_models_df = ranked_results.drop_duplicates(
    subset=["Category", "Commodity"],
    keep="first"
)

# Keep only the important columns
best_models_df = best_models_df[
    ["Category", "Commodity", "Model", "MAE", "RMSE"]
].reset_index(drop=True)

# Display results
print("Commodities with selected models:", len(best_models_df))
display(best_models_df.head(10).round(2))


Commodities with selected models: 58


,Category,Commodity,Model,MAE,RMSE
0,Cereals,"Corngrain [Maize] Yellow, matured",Holt-Winters,1.17,1.46
1,Fruits,"Banana Latundan, green",AutoARIMA,1.39,2.01
2,Fruits,"Banana Saba, green",AutoARIMA,2.00,2.74
3,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",Holt-Winters,2.12,2.65
4,Cereals,"Corngrain [Maize] White, matured",AutoARIMA,2.21,3.22
5,Fruits,"Banana Lakatan, green",AutoARIMA,2.39,3.42
6,Fruits,"Banana Bungulan, green",Holt-Winters,2.39,3.29
7,Fruit Vegetables,Upo [Bottle gourd],Holt-Winters,2.55,3.81
8,Fruit Vegetables,"Papaya, green",Holt-Winters,2.91,5.60
9,Fruits,Watermelon,Holt-Winters,2.98,4.26


In [ ]:

# Save the best-performing model for each commodity
best_models_df.to_csv(
    "../data/processed/best_models.csv",
    index=False
)

print("Best model selections saved successfully!")
print("Total commodities:", len(best_models_df))


In [72]:

# Generate future forecasts for a selected commodity
def forecast_commodity(data, category, commodity):

    # Prepare the selected commodity's monthly prices
    commodity_data = prepare_commodity_data(data, category, commodity)
    monthly_prices = commodity_data["Price"]

    # Find the best model and its evaluation metrics
    selected_result = best_models_df[
        (best_models_df["Category"] == category) &
        (best_models_df["Commodity"] == commodity)
    ].iloc[0]

    selected_model = selected_result["Model"]

    # Train the selected model using all historical prices
    current_model = forecasting_models[selected_model].clone()
    current_model.fit(monthly_prices)

    # Forecast the next 6 months
    predicted_prices = current_model.predict(fh=range(1, 7))

    # Organize the forecast results
    forecast_results = pd.DataFrame({
        "Category": category,
        "Commodity": commodity,
        "Last Observed Date": str(monthly_prices.index[-1]),
        "Last Observed Price": monthly_prices.iloc[-1],
        "Forecast Date": predicted_prices.index.astype(str),
        "Forecast Price": predicted_prices.values,
        "Model": selected_model,
        "MAE": selected_result["MAE"],
        "RMSE": selected_result["RMSE"]
    })

    return forecast_results



# Test using Palay
palay_forecast = forecast_commodity(
    model_data,
    "Cereals",
    "Palay [Paddy] Other Variety, dry (conv. to 14% mc)"
)

display(palay_forecast.round(2))


,Category,Commodity,Last Observed Date,Last Observed Price,Forecast Date,Forecast Price,Model,MAE,RMSE
0,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",2026-10,21.27,2026-11,21.47,Holt-Winters,2.12,2.65
1,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",2026-10,21.27,2026-12,21.84,Holt-Winters,2.12,2.65
2,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",2026-10,21.27,2027-01,22.27,Holt-Winters,2.12,2.65
3,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",2026-10,21.27,2027-02,22.65,Holt-Winters,2.12,2.65
4,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",2026-10,21.27,2027-03,22.74,Holt-Winters,2.12,2.65
5,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",2026-10,21.27,2027-04,22.84,Holt-Winters,2.12,2.65


In [ ]:

# Store future forecasts and any errors
all_forecasts = []
failed_forecasts = []

# Forecast all 58 commodities
for category, commodity in best_models_df[["Category", "Commodity"]].itertuples(index=False, name=None):
    try:
        # Generate the next six monthly forecasts
        forecast = forecast_commodity(model_data, category, commodity)

        all_forecasts.append(forecast)
        print(f"Completed: {commodity}")

    except Exception as error:
        failed_forecasts.append((category, commodity, str(error)))
        print(f"Failed: {commodity} - {error}")

# Combine forecasts into one DataFrame
all_forecasts_df = pd.concat(all_forecasts, ignore_index=True)

# Check the results
print("\nCommodities forecasted:", len(all_forecasts))
print("Failed commodities:", len(failed_forecasts))
print("Total forecasted prices:", len(all_forecasts_df))

display(all_forecasts_df.head(12).round(2))


In [75]:

# Sort forecasts by category, commodity, and date
all_forecasts_df = all_forecasts_df.sort_values(
    ["Category", "Commodity", "Forecast Date"]
).reset_index(drop=True)

# Save the forecast results
all_forecasts_df.to_csv(
    "../data/processed/future_six_month_forecasts.csv",
    index=False
)

# Verify the results
print("Forecasts saved successfully!")
print("Total forecasts:", len(all_forecasts_df))
print("Missing MAE/RMSE:", all_forecasts_df[["MAE", "RMSE"]].isna().sum().sum())

display(all_forecasts_df.head(12).round(2))


Forecasts saved successfully!
Total forecasts: 348
Missing MAE/RMSE: 0


,Category,Commodity,Last Observed Date,Last Observed Price,Forecast Date,Forecast Price,Model,MAE,RMSE
0,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-07,35.53,Holt-Winters,10.37,14.31
1,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-08,40.19,Holt-Winters,10.37,14.31
2,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-09,41.08,Holt-Winters,10.37,14.31
3,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-10,46.21,Holt-Winters,10.37,14.31
4,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-11,47.53,Holt-Winters,10.37,14.31
5,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-12,47.82,Holt-Winters,10.37,14.31
6,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-07,77.26,AutoARIMA,6.02,10.86
7,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-08,77.41,AutoARIMA,6.02,10.86
8,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-09,77.56,AutoARIMA,6.02,10.86
9,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-10,77.71,AutoARIMA,6.02,10.86


In [ ]:

# Count how many commodities selected each model
model_counts = best_models_df["Model"].value_counts()

print("Best Model Distribution:")
display(model_counts)

print("Total commodities:", model_counts.sum())


In [65]:

# Validate all future forecasts
print("Total forecasts:", len(all_forecasts_df))
print("Total commodities:", all_forecasts_df["Commodity"].nunique())

print("Missing values:", all_forecasts_df.isna().sum().sum())
print("Negative prices:", (all_forecasts_df["Forecast Price"] < 0).sum())

print("Duplicate forecasts:", all_forecasts_df.duplicated(
    subset=["Category", "Commodity", "Forecast Date"]
).sum())

# Verify six forecasts per commodity
forecast_counts = all_forecasts_df.groupby(
    ["Category", "Commodity"]
).size()

print("Incorrect forecast counts:", (forecast_counts != 6).sum())


Total forecasts: 348
Total commodities: 58
Missing values: 0
Negative prices: 0
Duplicate forecasts: 0
Incorrect forecast counts: 0


In [70]:

# Find the latest actual month for each commodity
latest_months = (
    model_data.groupby(["Category", "Commodity"])["Date"]
    .max().dt.to_period("M")
)

# Find the first forecast month for each commodity
first_forecast_months = (
    all_forecasts_df.groupby(["Category", "Commodity"])["Forecast Date"]
    .min().apply(lambda month: pd.Period(month, freq="M"))
)

# Verify forecasts start one month after the latest observation
incorrect_dates = (first_forecast_months != latest_months + 1).sum()

print("Commodities checked:", len(first_forecast_months))
print("Incorrect forecast starting dates:", incorrect_dates)


Commodities checked: 58
Incorrect forecast starting dates: 0


In [71]:

# Count selected models within each agricultural category
category_model_summary = pd.crosstab(
    best_models_df["Category"],
    best_models_df["Model"]
)

# Add the total number of commodities per category
category_model_summary["Total"] = category_model_summary.sum(axis=1)

display(category_model_summary)


Model,AutoARIMA,Holt-Winters,Seasonal Naive,Total
Category,,,,
Beans and Legumes,3,5,0,8
Cereals,2,2,0,4
Condiments,4,4,2,10
Fruit Vegetables,3,5,1,9
Fruits,5,6,0,11
Leafy Vegetables,5,5,0,10
Rootcrops,1,4,1,6


In [ ]:

import matplotlib.pyplot as plt

# Select Palay's future forecasts
palay_forecast = all_forecasts_df[
    all_forecasts_df["Commodity"] == "Palay [Paddy] Other Variety, dry (conv. to 14% mc)"
].copy()

# Convert forecast dates for plotting
palay_forecast["Forecast Date"] = pd.to_datetime(
    palay_forecast["Forecast Date"]
)

# Plot the last 24 months of actual prices
plt.figure(figsize=(10, 5))
plt.plot(
    palay_data["Price"].tail(24).index.to_timestamp(),
    palay_data["Price"].tail(24),
    label="Actual PSA Prices"
)

# Plot the six-month future forecast
plt.plot(
    palay_forecast["Forecast Date"],
    palay_forecast["Forecast Price"],
    marker="o",
    label="Six-Month Forecast"
)

plt.title("Palay - Historical Prices and Six-Month Forecast")
plt.xlabel("Month")
plt.ylabel("Farmgate Price (PHP/kg)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Import the Prophet forecasting model
from prophet import Prophet

# Prepare Lettuce data
lettuce_data = prepare_commodity_data(
    model_data, "Leafy Vegetables", "Lettuce"
)

_, _, _, lettuce_windows = prepare_train_test(lettuce_data)


# Prepare Lettuce prices for Prophet
prophet_data = pd.DataFrame({
    "ds": lettuce_data.index.to_timestamp(),
    "y": lettuce_data["Price"].values
})

print(prophet_data.head())
print("\nTotal observations:", len(prophet_data))

In [ ]:
# Select the first 6-month testing window
test_window = lettuce_windows[0]

# Use only historical prices before the testing period
training_data = prophet_data[
    prophet_data["ds"] < test_window.index[0].to_timestamp()
]

# Create and train the Prophet model
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

prophet_model.fit(training_data)

# Predict prices for the six testing months
future_dates = pd.DataFrame({
    "ds": test_window.index.to_timestamp()
})

predictions = prophet_model.predict(future_dates)

print(predictions[["ds", "yhat"]])

In [ ]:
# Get actual and predicted prices
actual_prices = test_window.values
predicted_prices = predictions["yhat"].values

# Calculate forecasting errors
prophet_mae = mean_absolute_error(actual_prices, predicted_prices)
prophet_rmse = np.sqrt(mean_squared_error(actual_prices, predicted_prices))

print("Prophet MAE:", round(prophet_mae, 2))
print("Prophet RMSE:", round(prophet_rmse, 2))

In [85]:
# Store the evaluation results
prophet_results = []

# Test Prophet on all six backtesting windows
for window_number, test_window in enumerate(lettuce_windows, start=1):

    # Use only prices before the testing window
    training_data = prophet_data[
        prophet_data["ds"] < test_window.index[0].to_timestamp()
    ]

    # Train a new Prophet model
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False
    )

    model.fit(training_data)

    # Predict the six testing months
    future_dates = pd.DataFrame({
        "ds": test_window.index.to_timestamp()
    })

    predictions = model.predict(future_dates)["yhat"].values

    # Calculate errors
    mae = mean_absolute_error(test_window.values, predictions)
    rmse = np.sqrt(mean_squared_error(test_window.values, predictions))

    prophet_results.append({
        "Window": window_number,
        "MAE": mae,
        "RMSE": rmse
    })

# Display the evaluation results
prophet_results_df = pd.DataFrame(prophet_results)

print(prophet_results_df.round(2))

print("\nOverall Prophet MAE:",
      round(prophet_results_df["MAE"].mean(), 2))

print("Overall Prophet RMSE:",
      round(np.sqrt(np.mean(prophet_results_df["RMSE"] ** 2)), 2))

15:17:12 - cmdstanpy - INFO - Chain [1] start processing
15:17:12 - cmdstanpy - INFO - Chain [1] done processing
15:17:12 - cmdstanpy - INFO - Chain [1] start processing
15:17:12 - cmdstanpy - INFO - Chain [1] done processing
15:17:12 - cmdstanpy - INFO - Chain [1] start processing
15:17:12 - cmdstanpy - INFO - Chain [1] done processing
15:17:12 - cmdstanpy - INFO - Chain [1] start processing
15:17:13 - cmdstanpy - INFO - Chain [1] done processing
15:17:13 - cmdstanpy - INFO - Chain [1] start processing
15:17:13 - cmdstanpy - INFO - Chain [1] done processing
15:17:13 - cmdstanpy - INFO - Chain [1] start processing
15:17:13 - cmdstanpy - INFO - Chain [1] done processing


   Window    MAE    RMSE
0       1   6.55    7.37
1       2  34.45   44.44
2       3  68.52   78.98
3       4  82.48  105.65
4       5  46.37   52.06
5       6  51.86   69.25

Overall Prophet MAE: 48.37
Overall Prophet RMSE: 67.0
